# 04 – Train the visual vocabularies

This notebook fits one MiniBatchKMeans vocabulary for each descriptor configuration. All fitting uses descriptors from the 391 gallery images only.

Fixed settings: K = 100, maximum 200 training descriptors per gallery image, random seed 3032.

# Project setup

This notebook uses the shared functions in `visual_search/`.

In [ ]:
from pathlib import Path
import sys

# Mount Google Drive when this notebook is running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

PROJECT_ROOT = Path("/content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3")
DATASET_SOURCE = Path("/content/drive/MyDrive/cvpr_coursework/MSRC_ObjCategImageDatabase_v2")

if not (PROJECT_ROOT / "cvpr").is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to the folder containing cvpr/ and visual_search/.")
if not (PROJECT_ROOT / "visual_search").is_dir():
    raise FileNotFoundError("The visual_search package is missing from PROJECT_ROOT.")

sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)

Mounted at /content/drive
Project: /content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from cvpr.dataset import load_split_records, read_rgb
from cvpr.baseline import global_rgb_histogram
from cvpr.retrieval import compute_descriptors
from cvpr.evaluation import evaluate_retrieval, per_category_summary
from cvpr.plotting import show_ranked_results

from visual_search.features import create_sift
from visual_search.cache import feature_cache_path, load_feature_cache

MANIFEST = PROJECT_ROOT / "split_manifest.csv"
gallery, dev, test = load_split_records(
    DATASET_SOURCE,
    manifest_path=MANIFEST,
    extract_dir=PROJECT_ROOT / "data",
)
assert (len(gallery), len(dev), len(test)) == (391, 100, 100)
print(f"Loaded {len(gallery)} gallery, {len(dev)} development and {len(test)} test images.")

Loaded 391 gallery, 100 development and 100 test images.


In [ ]:
import csv
from visual_search.vocabulary import (
    VOCABULARY_CONFIGS, train_all_vocabularies, vocabulary_path,
    save_vocabulary_summary,
)

K = 100
DESCRIPTOR_CAP = 200
RANDOM_SEED = 3032
VOCABULARY_DIR = PROJECT_ROOT / "bovw_vocabularies"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("Configurations:", len(VOCABULARY_CONFIGS))
print("K:", K)
print("Maximum descriptors per gallery image:", DESCRIPTOR_CAP)
print("Random seed:", RANDOM_SEED)

Configurations: 8
K: 100
Maximum descriptors per gallery image: 200
Random seed: 3032


## 1. Check the input caches

Each of the eight configurations must have one cache for every gallery image. SIFT and RootSIFT use the same raw SIFT cache; RootSIFT is applied when the training matrix is built.

In [ ]:
from visual_search.vocabulary import descriptor_cache_folder
from visual_search.cache import feature_cache_path, load_feature_cache

for config in VOCABULARY_CONFIGS:
    folder = descriptor_cache_folder(PROJECT_ROOT, config)
    missing = []
    for index, record in enumerate(gallery):
        if not feature_cache_path(folder, index, record).exists():
            missing.append(record.filename)
    print(f"{config.name:32s} missing: {len(missing)}")
    if missing:
        raise FileNotFoundError(f"Missing caches for {config.name}: {missing[:3]}")

detected_sift                    missing: 0
detected_rootsift                missing: 0
detected_opponentsift            missing: 0
detected_root_opponentsift       missing: 0
dense_sift                       missing: 0
dense_rootsift                   missing: 0
dense_opponentsift               missing: 0
dense_root_opponentsift          missing: 0


## 2. Train or reuse all eight vocabularies

A separate vocabulary is required for each descriptor representation. The 128-D and 384-D descriptors cannot share cluster centres.

In [ ]:
vocabulary_results = train_all_vocabularies(
    gallery,
    PROJECT_ROOT,
    configs=VOCABULARY_CONFIGS,
    k=K,
    maximum_per_image=DESCRIPTOR_CAP,
    random_seed=RANDOM_SEED,
    batch_size=2048,
    max_iter=100,
    n_init=3,
    rebuild=False,
)

summary_path = save_vocabulary_summary(
    vocabulary_results, RESULTS_DIR / "stage4_vocabulary_summary.csv"
)
print("Summary saved to:", summary_path)

[1/8] detected_sift
  reused: (100, 128), 75631 training descriptors, 0.99s, 0 empty clusters
[2/8] detected_rootsift
  reused: (100, 128), 75631 training descriptors, 2.31s, 0 empty clusters
[3/8] detected_opponentsift
  reused: (100, 384), 75631 training descriptors, 2.31s, 0 empty clusters
[4/8] detected_root_opponentsift
  reused: (100, 384), 75631 training descriptors, 2.41s, 0 empty clusters
[5/8] dense_sift
  reused: (100, 128), 78200 training descriptors, 0.93s, 0 empty clusters
[6/8] dense_rootsift
  reused: (100, 128), 78200 training descriptors, 0.88s, 0 empty clusters
[7/8] dense_opponentsift
  reused: (100, 384), 78200 training descriptors, 2.30s, 0 empty clusters
[8/8] dense_root_opponentsift
  reused: (100, 384), 78200 training descriptors, 2.21s, 0 empty clusters
Summary saved to: /content/drive/MyDrive/cvpr_coursework/EEE3032_Coursework_Starter_v3/results/stage4_vocabulary_summary.csv


## 3. Verify the saved vocabulary shapes

In [ ]:
for config in VOCABULARY_CONFIGS:
    path = vocabulary_path(PROJECT_ROOT, config, K, DESCRIPTOR_CAP, RANDOM_SEED)
    with np.load(path, allow_pickle=False) as saved:
        centres = saved["centres"]
        cluster_sizes = saved["cluster_sizes"]
    assert centres.shape == (K, config.dimension)
    assert np.isfinite(centres).all()
    print(
        f"{config.name:32s} shape={centres.shape}, "
        f"empty clusters={(cluster_sizes == 0).sum()}"
    )

detected_sift                    shape=(100, 128), empty clusters=0
detected_rootsift                shape=(100, 128), empty clusters=0
detected_opponentsift            shape=(100, 384), empty clusters=0
detected_root_opponentsift       shape=(100, 384), empty clusters=0
dense_sift                       shape=(100, 128), empty clusters=0
dense_rootsift                   shape=(100, 128), empty clusters=0
dense_opponentsift               shape=(100, 384), empty clusters=0
dense_root_opponentsift          shape=(100, 384), empty clusters=0
